# 23 · The ladder on clean labels

**Purpose.** On A2, the 107 pairs where the two humans split (the LLM broke the tie, D41) cap
every score. This notebook asks whether the **order of the ladder rungs** changes when those pairs
are dropped. If the sophisticated rungs gain more on clean pairs, label noise was hiding their
value.

**Status.** Reporting only. "Split pairs dropped" is the existing `llm_free` subset (tiers
`humans_agree` + `single_human`); A2 has no `llm_only` pairs. Every figure is read from committed
records (`stage1-metrics.json`, `stage2-metrics.json`); nothing is fitted or re-scored here. AUC is
the mean per-JD AUC, Fit vs No Fit (D47), with JD-resampled 95% CIs.

In [1]:
import json

import pandas as pd

from candidate_screener.config import OUTPUT

s1 = json.load(open(OUTPUT / "matching" / "stage1-metrics.json"))
s2 = json.load(open(OUTPUT / "matching" / "stage2-metrics.json"))
x4x2 = json.load(open(OUTPUT / "matching" / "ladder-clean-labels.json"))
fmt = lambda v, lo, hi: f"{v:.3f} [{lo:.2f}, {hi:.2f}]"

## 1 · Every rung, all pairs beside clean pairs

In [2]:
rows = {}
for name, s in s1["systems"].items():
    for f in s["figures"]:
        if f["metric"] == "auc":
            rows.setdefault(f"{s['rung']} {name}", {})[(f["subset"], f["scope"])] = f["value"]
t = pd.DataFrame(rows).T[[(a, b) for a in ("all", "llm_free") for b in ("generic", "targeted", "total")]]
t[("gain", "total")] = t[("llm_free", "total")] - t[("all", "total")]
t.sort_values(("all", "total"), ascending=False).round(3)

all                 llm_free  \
                                            generic targeted  total  generic   
X4 extracted rule + SBERT (weak fine-tuned)   0.891    0.785  0.855    0.912   
X2 extracted rule + TF-IDF                    0.863    0.826  0.850    0.893   
1a+2 metadata + TF-IDF                        0.843    0.851  0.846    0.876   
3b SBERT (weak fine-tuned)                    0.853    0.796  0.834    0.880   
X3 extracted rule + skill overlap             0.851    0.761  0.821    0.873   
X1 extracted rule                             0.851    0.755  0.818    0.875   
2 TF-IDF                                      0.780    0.807  0.789    0.810   
1a metadata rules                             0.776    0.784  0.778    0.805   
3a SBERT (zero-shot)                          0.740    0.731  0.737    0.768   
0 random                                      0.558    0.552  0.556    0.569   

                                                              gain  
                                            targeted  total  total  
X4 extracted rule + SBERT (weak fine-tuned)    0.845  0.892  0.037  
X2 extracted rule + TF-IDF                     0.917  0.900  0.050  
1a+2 metadata + TF-IDF                         0.819  0.859  0.013  
3b SBERT (weak fine-tuned)                     0.789  0.852  0.018  
X3 extracted rule + skill overlap              0.841  0.863  0.042  
X1 extracted rule                              0.847  0.866  0.048  
2 TF-IDF                                       0.860  0.826  0.036  
1a metadata rules                              0.784  0.799  0.020  
3a SBERT (zero-shot)                           0.786  0.774  0.037  
0 random                                       0.628  0.587  0.031

Pairs counted: `all` 590 (generic 390, targeted 200), `llm_free` 460 (generic 350, targeted 110).
Ten pairs sit in JDs without both classes.

## 2 · The registered comparisons (K1-K5), paired, on both subsets

In [3]:
rows = {}
for c in s1["comparisons"]:
    for d in c["deltas"]:
        if d["metric"] == "auc":
            rows.setdefault((f"{c['id']}: {c['b']} vs {c['a']}", d["subset"]), {})[d["scope"]] = fmt(d["delta"], *d["ci"])
pd.DataFrame(rows).T[["generic", "targeted", "total"]]

generic  \
K1: extracted rule + TF-IDF vs TF-IDF              all         0.082 [0.04, 0.13]   
                                                   llm_free    0.083 [0.04, 0.13]   
K2: extracted rule + TF-IDF vs metadata + TF-IDF   all        0.020 [-0.04, 0.08]   
                                                   llm_free   0.017 [-0.05, 0.09]   
K3: extracted rule + SBERT (weak fine-tuned) vs... all         0.037 [0.00, 0.07]   
                                                   llm_free   0.032 [-0.00, 0.07]   
K4: extracted rule + TF-IDF vs SBERT (weak fine... all        0.009 [-0.04, 0.06]   
                                                   llm_free   0.013 [-0.04, 0.06]   
K5: extracted rule + skill overlap vs extracted... all        0.001 [-0.02, 0.02]   
                                                   llm_free  -0.002 [-0.05, 0.03]   

                                                                         targeted  \
K1: extracted rule + TF-IDF vs TF-IDF              all        0.019 [-0.02, 0.06]   
                                                   llm_free    0.057 [0.02, 0.10]   
K2: extracted rule + TF-IDF vs metadata + TF-IDF   all       -0.025 [-0.07, 0.01]   
                                                   llm_free    0.098 [0.05, 0.15]   
K3: extracted rule + SBERT (weak fine-tuned) vs... all       -0.011 [-0.06, 0.03]   
                                                   llm_free    0.056 [0.01, 0.12]   
K4: extracted rule + TF-IDF vs SBERT (weak fine... all        0.030 [-0.04, 0.11]   
                                                   llm_free    0.128 [0.05, 0.23]   
K5: extracted rule + skill overlap vs extracted... all        0.006 [-0.05, 0.06]   
                                                   llm_free  -0.006 [-0.06, 0.06]   

                                                                            total  
K1: extracted rule + TF-IDF vs TF-IDF              all         0.061 [0.03, 0.10]  
                                                   llm_free    0.075 [0.04, 0.11]  
K2: extracted rule + TF-IDF vs metadata + TF-IDF   all        0.005 [-0.03, 0.05]  
                                                   llm_free   0.042 [-0.01, 0.09]  
K3: extracted rule + SBERT (weak fine-tuned) vs... all        0.021 [-0.01, 0.05]  
                                                   llm_free    0.040 [0.01, 0.07]  
K4: extracted rule + TF-IDF vs SBERT (weak fine... all        0.016 [-0.02, 0.06]  
                                                   llm_free    0.048 [0.00, 0.10]  
K5: extracted rule + skill overlap vs extracted... all        0.002 [-0.02, 0.03]  
                                                   llm_free  -0.003 [-0.04, 0.03]

### 2.1 · The comparison the stage-1 record lacks: X4 vs X2

`clean_labels.py` re-scores both rungs exactly as stage 1 did (score digests checked against the
record) and adds the paired delta. Reporting only.

In [4]:
pd.DataFrame({(d["subset"], d["scope"]): {"Δ AUC (X4 − X2)": fmt(d["delta"], *d["ci"]), "n JDs": d["n_jd"]}
              for d in x4x2["deltas"]}).T

Δ AUC (X4 − X2) n JDs
all      generic     0.028 [0.00, 0.06]    39
         targeted  -0.041 [-0.11, 0.02]    20
         total      0.004 [-0.03, 0.03]    59
llm_free generic    0.020 [-0.01, 0.05]    38
         targeted  -0.072 [-0.19, 0.04]    17
         total     -0.009 [-0.05, 0.03]    55

## 3 · The stage-2 rerankers (ordering on the human grades, D52)

R1 and R2 are cross-encoders, R3 a bi-encoder, all fine-tuned from MiniLM on the D50 batch's
LLM scores (D50/D51). Stage 2 reports nDCG and pairwise agreement on the human grades (mean over the two annotators),
not AUC. `full` is every pair with a human grade; `llm_free` drops the split pairs.

In [5]:
rows = {}
for name, s in s2["systems"].items():
    for f in s["figures"]:
        if f.get("reference") == "mean" and f["subset"] in ("full", "llm_free"):
            rows.setdefault(name, {})[(f["metric"], f["subset"], f["scope"])] = f["value"]
t2 = pd.DataFrame(rows).T
t2 = t2[[(m, s, sc) for m in ("pairwise", "ndcg") for s in ("full", "llm_free") for sc in ("generic", "targeted", "total")]]
t2.sort_values(("pairwise", "full", "total"), ascending=False).round(3)

pairwise                                           \
                            full                 llm_free                   
                         generic targeted  total  generic targeted  total   
LLM score (0-100)          0.783    0.748  0.770    0.862    0.845  0.855   
SBERT (weak fine-tuned)    0.762    0.681  0.732    0.815    0.734  0.781   
R3                         0.757    0.685  0.730    0.837    0.769  0.808   
TF-IDF                     0.744    0.691  0.724    0.780    0.813  0.794   
SBERT (zero-shot)          0.701    0.676  0.692    0.764    0.753  0.760   
R1                         0.628    0.596  0.616    0.683    0.621  0.657   
R2                         0.618    0.573  0.601    0.693    0.499  0.610   
random (expected)          0.500    0.500  0.500    0.500    0.500  0.500   

                           ndcg                                           
                           full                 llm_free                  
                        generic targeted  total  generic targeted  total  
LLM score (0-100)         0.954    0.874  0.924    0.982    0.926  0.958  
SBERT (weak fine-tuned)   0.950    0.861  0.916    0.965    0.884  0.931  
R3                        0.959    0.848  0.917    0.980    0.907  0.949  
TF-IDF                    0.951    0.859  0.916    0.955    0.914  0.938  
SBERT (zero-shot)         0.944    0.863  0.913    0.964    0.905  0.939  
R1                        0.897    0.800  0.860    0.911    0.849  0.885  
R2                        0.910    0.785  0.863    0.927    0.769  0.859  
random (expected)         0.869    0.769  0.831    0.871    0.782  0.833

### 3.1 · Every reranker against X2 (not registered, `exploratory`)

Stage 2 compared the rerankers with SBERT weak and TF-IDF only. X2 never saw an LLM score (no
training, no LLM in its rule), so it is an independent yardstick for models trained on the D50
LLM scores. Paired per-JD pairwise agreement minus X2's, on the human grades (D52): reference `yc`
(all JDs) and `mean` (where both humans graded). Every score digest is checked against the
committed stage-1 and stage-2 records (`clean_labels.py`).

In [6]:
o = x4x2["ordering_vs_x2"]
rows = {}
for d in o["deltas"]:
    if d["metric"] == "pairwise":
        rows.setdefault((o["names"].get(d["b"], d["b"]), d["reference"]), {})[(d["subset"], d["scope"])] = fmt(d["delta"], *d["ci"])
pd.DataFrame(rows).T

full  \
                                                             generic   
cross-encoder (LLM batch, pointwise)     yc    -0.163 [-0.24, -0.09]   
                                         mean   -0.131 [-0.30, 0.04]   
cross-encoder (LLM batch, pairwise)      yc    -0.165 [-0.24, -0.09]   
                                         mean   -0.140 [-0.31, 0.03]   
bi-encoder (LLM batch)                   yc      0.025 [-0.02, 0.07]   
                                         mean   -0.001 [-0.13, 0.12]   
SBERT (weak fine-tuned)                  yc      0.014 [-0.03, 0.06]   
                                         mean    0.004 [-0.14, 0.15]   
extracted rule + SBERT (weak fine-tuned) yc      0.019 [-0.01, 0.05]   
                                         mean   -0.039 [-0.16, 0.06]   

                                                                      \
                                                            targeted   
cross-encoder (LLM batch, pointwise)     yc    -0.269 [-0.34, -0.20]   
                                         mean  -0.125 [-0.21, -0.02]   
cross-encoder (LLM batch, pairwise)      yc    -0.277 [-0.39, -0.18]   
                                         mean  -0.148 [-0.25, -0.04]   
bi-encoder (LLM batch)                   yc     -0.076 [-0.15, 0.00]   
                                         mean   -0.037 [-0.11, 0.04]   
SBERT (weak fine-tuned)                  yc    -0.057 [-0.11, -0.00]   
                                         mean   -0.040 [-0.09, 0.01]   
extracted rule + SBERT (weak fine-tuned) yc     -0.015 [-0.06, 0.03]   
                                         mean   -0.018 [-0.06, 0.03]   

                                                                      \
                                                               total   
cross-encoder (LLM batch, pointwise)     yc    -0.197 [-0.25, -0.14]   
                                         mean  -0.129 [-0.24, -0.02]   
cross-encoder (LLM batch, pairwise)      yc    -0.201 [-0.27, -0.14]   
                                         mean  -0.143 [-0.26, -0.03]   
bi-encoder (LLM batch)                   yc     -0.007 [-0.05, 0.04]   
                                         mean   -0.014 [-0.10, 0.07]   
SBERT (weak fine-tuned)                  yc     -0.009 [-0.05, 0.03]   
                                         mean   -0.013 [-0.11, 0.08]   
extracted rule + SBERT (weak fine-tuned) yc      0.008 [-0.02, 0.03]   
                                         mean   -0.031 [-0.10, 0.03]   

                                                            llm_free  \
                                                             generic   
cross-encoder (LLM batch, pointwise)     yc    -0.161 [-0.24, -0.09]   
                                         mean   -0.107 [-0.32, 0.09]   
cross-encoder (LLM batch, pairwise)      yc    -0.159 [-0.24, -0.08]   
                                         mean   -0.098 [-0.31, 0.11]   
bi-encoder (LLM batch)                   yc      0.025 [-0.03, 0.07]   
                                         mean    0.046 [-0.10, 0.19]   
SBERT (weak fine-tuned)                  yc      0.017 [-0.04, 0.07]   
                                         mean    0.025 [-0.15, 0.20]   
extracted rule + SBERT (weak fine-tuned) yc      0.017 [-0.01, 0.05]   
                                         mean   -0.013 [-0.14, 0.09]   

                                                                      \
                                                            targeted   
cross-encoder (LLM batch, pointwise)     yc    -0.218 [-0.36, -0.08]   
                                         mean  -0.223 [-0.36, -0.08]   
cross-encoder (LLM batch, pairwise)      yc    -0.362 [-0.53, -0.21]   
                                         mean  -0.345 [-0.49, -0.21]   
bi-encoder (LLM batch)                   yc     -0.060 [-0.19, 0.06]   
                                         mean   -0.075 [-0.21, 0.05]   
SBERT (weak fine-tuned)     

X2's and X4's own pairwise agreement (reference `mean`), for scale against §3's table:

In [7]:
pd.DataFrame({name: {(f["subset"], f["scope"]): f["value"] for f in figs
                     if f["reference"] == "mean" and f["metric"] == "pairwise" and f["subset"] in ("full", "llm_free")}
              for name, figs in o["x2_x4_figures"].items()}).T.round(3)

full                 llm_free  \
                                         generic targeted  total  generic   
extracted rule + SBERT (weak fine-tuned)   0.719    0.703  0.713    0.778   
extracted rule + TF-IDF                    0.758    0.721  0.744    0.791   

                                                          
                                         targeted  total  
extracted rule + SBERT (weak fine-tuned)    0.810  0.791  
extracted rule + TF-IDF                     0.844  0.813

## 4 · Reading

1. **X4 and X2 are level on both subsets.** Paired, X4 (rule + fine-tuned SBERT) minus X2 (rule +
   TF-IDF): +0.004 [−0.026, +0.033] on all pairs, −0.009 [−0.054, +0.032] on clean pairs. X4 is
   ahead on generic (+0.028 resolved on all pairs), X2 on targeted (−0.041, −0.072, CIs cross 0).
   Adding SBERT in place of TF-IDF does not earn its cost on either subset.
2. **The gaps widen in favour of the simple hybrid.** K4, X2 vs the fine-tuned SBERT: +0.016
   [−0.025, +0.058] on all pairs, **+0.048 [+0.001, +0.097]** on clean pairs (targeted +0.128).
   K1 (X2 vs TF-IDF) also widens: +0.061 → +0.075.
3. **The fine-tuned SBERT gains least from clean labels** (0.834 → 0.852 total), the lexical and
   rule rungs most (X2 +0.050, TF-IDF +0.037).
4. **The cross-encoders stay at the bottom.** On pairwise agreement with the human grades, R1 and
   R2 (cross-encoders) are below TF-IDF and both SBERTs on both subsets (R2 total 0.601 / 0.610). The
   bi-encoder R3 is level with the fine-tuned SBERT on all pairs (0.730 vs 0.732) and ahead on
   clean pairs (0.808 vs 0.781, unpaired), edging TF-IDF there (0.794). Stage 2 already judged R3
   vs SBERT weak "not resolved" and adopted SBERT weak; this does not change the selection (D48).

5. **Against X2, no reranker wins (§3.1).** The cross-encoders are clearly worse than X2 on every
   cell (R2 pairwise −0.20 [−0.27, −0.14] vs `yc`, total, all pairs; −0.22 on clean pairs). The
   bi-encoder R3 and SBERT weak are level with X2 (R3 −0.007 [−0.05, +0.04]; clean −0.001).
   R3 is ahead of X2 on generic nDCG (+0.043 [+0.01, +0.08]), the one resolved gain, and behind
   on targeted. A rule + TF-IDF score that never saw an LLM label orders candidates as well as
   the best model trained on 1,000 LLM-scored pairs.

**For the research question:** label noise was not hiding the value of more sophisticated
matching. On the cleaner labels, the simple rule + TF-IDF hybrid's lead over the fine-tuned SBERT
is larger, not smaller, and resolves (K4); X4 does not beat X2 on either subset; the cross-encoders
stay worst and the fine-tuned bi-encoder R3 gains a little ground, not enough to change the picture.
Caveat: `llm_free` drops the hardest, most ambiguous pairs as well as the noisiest, so it is an
easier test, not only a cleaner one; and targeted keeps only 110 pairs, so its CIs are wide.